# M2 - EDA & Feature Engineering (Regression)
**IT3051 FDM Mini Project 2026 - DataCo Smart Supply Chain**  |  Owner: M2

**Task:** regression. **Target:** `Days for shipping (real)` (whole days, 0-6), predicted at order placement.

**Golden rule:** all EDA uses the **training set only**. The test set is loaded once, in Step 10, and is only *transformed*.

**Before running:** run M1's notebook first so `data/processed/train_reg.csv` and `test_reg.csv` exist.

After every step there is an *Observation* cell. Fill it in **in your own words, with the numbers you actually see** - that is what you say in the viva.

## Step 0 - Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))                       # so "from src.features import ..." works
PROCESSED_DIR = ROOT / "data" / "processed"
FIG_DIR = ROOT / "reports" / "figures" / "m2"
FIG_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "Days for shipping (real)"
GROUP = "Order Id"
DATE_COL = "order date (DateOrders)"
DATE_FMT = "%m/%d/%Y %H:%M"

def show(fig, name):
    # Tidy, save for the report, and display
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"{name}.png", dpi=120)
    plt.show()

**Viva point:** each chart is saved with a numbered name (`01_...`, `02_...`), so every figure in the report links back to the code that made it.

## Step 1 - Load the training data (train only)

In [ ]:
train = pd.read_csv(PROCESSED_DIR / "train_reg.csv")
train["order_dt"] = pd.to_datetime(train[DATE_COL], format=DATE_FMT)   # EDA helper column only, not a feature
y = train[TARGET]
print(f"Train: {train.shape[0]:,} rows, {train[GROUP].nunique():,} orders, {train.shape[1] - 1} original columns")
print(f"Order dates: {train['order_dt'].min()}  ->  {train['order_dt'].max()}")

**Expected (from M1's split summary):** about 138,098 rows and 50,318 orders.

**Viva point - "Why not EDA on the full data?"** The test set must stay unseen until the final evaluation. Every decision here (which features to build, which outliers to treat) is based on training data only.

## Step 2 - Structure of the data

### 2.1 Column types and cardinality

In [ ]:
# 'Category Id' is a label code (1 code = 1 category), not a quantity, so it is not treated as numeric.
ID_CODES = ["Category Id"]

NUMERIC = [c for c in train.select_dtypes("number").columns if c not in (TARGET, GROUP, *ID_CODES)]
CATEGORICAL = [c for c in train.select_dtypes(exclude="number").columns if c not in (DATE_COL, "order_dt")]

structure = pd.DataFrame({
    "type": ["numeric" if c in NUMERIC else "categorical" for c in NUMERIC + CATEGORICAL],
    "n_unique": [train[c].nunique() for c in NUMERIC + CATEGORICAL],
    "example": [train[c].iloc[0] for c in NUMERIC + CATEGORICAL],
}, index=NUMERIC + CATEGORICAL)
structure

**Viva point:** the number of unique values decides the encoding - low-cardinality columns can be one-hot encoded, high-cardinality ones (Order City, Order State...) need M3's special encoding.
Note that `Latitude` / `Longitude` are the **customer's** location, not the delivery destination.

### 2.2 Summary statistics of numeric columns

In [ ]:
train[NUMERIC].describe().T.round(2)

**Viva point:** negative minimums and long right tails here are the first sign of the skew and outliers investigated in Step 7.

**Observation (write your own):**

## Step 3 - Missing values and duplicates

In [ ]:
missing = train.isna().sum()
print("Columns with missing values:", missing[missing > 0].to_dict() or "none")
print("Exact duplicate rows:", train.duplicated().sum())

items = train.groupby(GROUP).size()
print(f"\nRows per order: mean {items.mean():.2f}, max {items.max()}, orders with >1 item {(items > 1).mean() * 100:.1f}%")

**Viva point - "Are there duplicates?"** No duplicate rows. Orders repeat across rows because each row is one *item*; that is not duplication, and it is why the split is grouped by Order Id.

**Observation (write your own):**

## Step 4 - The target

### 4.1 Distribution of real shipping days

In [ ]:
print(y.describe().round(2))
print(f"Skewness: {y.skew():.2f}")

fig, ax = plt.subplots(figsize=(6, 3))
y.value_counts().sort_index().plot.bar(ax=ax)
ax.set_xlabel("Real shipping days"); ax.set_ylabel("Rows"); ax.set_title("Distribution of real shipping days")
show(fig, "01_target_distribution")

**Viva point:** the target is a whole number from 0 to 6 with no heavy tail, so no log transform is needed. It is discrete but ordered, which suits regression, and MAE is in *days*, easy to explain.

### 4.2 Shipping days by shipping mode

In [ ]:
by_mode = train.groupby("Shipping Mode")[TARGET].agg(["mean", "std", "min", "max", "count"]).sort_values("mean")
print(by_mode.round(2))

fig, ax = plt.subplots(figsize=(6, 3))
by_mode["mean"].plot.barh(ax=ax, xerr=by_mode["std"])
ax.set_xlabel("Mean real shipping days (bar = +/-1 std)"); ax.set_title("Shipping days by shipping mode")
show(fig, "02_days_by_shipping_mode")

**Viva point:**
- *Between* modes: the difference in the means is what Shipping Mode explains.
- *Within* a mode: the spread (error bars) is what the other features must explain. Wide bars mean Shipping Mode alone leaves a lot of error.

**Observation (write your own):**

## Step 5 - Categorical features vs the target

### 5.1 Strength of each categorical feature (eta squared)

In [ ]:
def eta_squared(cat, target):
    # Share of the target's variance explained by the category groups (0 = none, 1 = all)
    overall = target.mean()
    g = target.groupby(cat)
    ss_between = (g.size() * (g.mean() - overall) ** 2).sum()
    ss_total = ((target - overall) ** 2).sum()
    return ss_between / ss_total

cat_strength = pd.DataFrame({
    "n_categories": [train[c].nunique() for c in CATEGORICAL],
    "eta_squared": [eta_squared(train[c], y) for c in CATEGORICAL],
}, index=CATEGORICAL).sort_values("eta_squared", ascending=False)
cat_strength.round(4)

**Viva point:**
- *Why eta squared:* Pearson only works between two numbers; eta squared is the matching measure for a category vs a numeric target.
- *Caution:* columns with thousands of categories (Order City, Order State) get an inflated value by chance, because each tiny group has its own average. That is why `n_categories` is shown, and why M3 must judge them with cross-validation.
- *Cross-check:* M1's "Shipping Mode only" baseline reached CV R2 = 0.389, so Shipping Mode's eta squared should be close to 0.39.

### 5.2 Shipping days by market

In [ ]:
by_market = train.groupby("Market")[TARGET].mean().sort_values()
fig, ax = plt.subplots(figsize=(6, 3))
by_market.plot.bar(ax=ax)
ax.set_ylabel("Mean real shipping days"); ax.set_title("Shipping days by market")
ax.set_ylim(0, by_market.max() * 1.2)
show(fig, "03_days_by_market")

**Viva point:** if the markets look the same, the destination region explains little of the shipping time - a finding that tells M3 the geographic columns are candidates for removal.

### 5.3 Segment, payment type and department

In [ ]:
for col in ["Customer Segment", "Type", "Department Name"]:
    print(train.groupby(col)[TARGET].agg(["mean", "count"]).round(2), "\n")

**Viva point:** weak relationships are evidence too. "Customer segment does not affect shipping time" is as useful as a strong effect.

**Observation (write your own):**

## Step 6 - Numeric features vs the target, and correlation

### 6.1 Correlation of each numeric column with the target

In [ ]:
num_vs_target = pd.DataFrame({
    "pearson": train[NUMERIC].corrwith(y),
    "spearman": train[NUMERIC].corrwith(y, method="spearman"),
}).sort_values("spearman", key=abs, ascending=False)
num_vs_target.round(3)

**Viva point:** Pearson measures straight-line relationships; Spearman uses ranks, so it also catches curved relationships and resists outliers. Near-zero values do not prove a feature is useless (trees can find interactions), but they suggest a small individual effect.

### 6.2 Correlation heatmap

In [ ]:
corr = train[NUMERIC + [TARGET]].corr()
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=90, fontsize=8)
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns, fontsize=8)
fig.colorbar(im, ax=ax); ax.set_title("Correlation of numeric columns")
show(fig, "04_correlation_heatmap")

**Viva point:** strongly related inputs (multicollinearity) make linear-regression coefficients unstable; tree models are much less affected.

### 6.3 Highly correlated pairs (|r| > 0.9)

In [ ]:
pairs = corr.drop(index=TARGET, columns=TARGET).stack()
pairs = pairs[[a < b for a, b in pairs.index]]                # each pair once
high = pairs[pairs.abs() > 0.9].sort_values(key=abs, ascending=False)
print("Numeric pairs with |r| > 0.9:")
print(high.round(3) if len(high) else "none")

**Viva point:** this list goes to M3 for feature selection - for the linear model keep only one column from each pair.

**Observation (write your own):**

## Step 7 - Outliers, skew and consistency checks

### 7.1 Outlier share and skewness per column

In [ ]:
def iqr_outliers(s):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return ((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).mean() * 100

outliers = pd.DataFrame({
    "outliers_%": [iqr_outliers(train[c]) for c in NUMERIC],
    "skew": [train[c].skew() for c in NUMERIC],
    "min": [train[c].min() for c in NUMERIC],
    "max": [train[c].max() for c in NUMERIC],
}, index=NUMERIC).sort_values("outliers_%", ascending=False)
outliers.round(2)

**Viva point:** IQR rule = a value is an outlier if it lies more than 1.5 x IQR beyond the 25th/75th percentile. This table is the evidence M4 uses for outlier/skew treatment.

### 7.2 Boxplot of the financial columns

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
ax.boxplot([train["Benefit per order"], train["Sales"]], vert=False)
ax.set_yticks([1, 2]); ax.set_yticklabels(["Benefit per order", "Sales"])
ax.axvline(0, color="grey", lw=0.8); ax.set_title("Outliers in the financial columns")
show(fig, "05_boxplot_financial")

### 7.3 Negative profits - errors or real losses?

In [ ]:
neg = train["Benefit per order"] < 0
print(f"Rows with negative Benefit per order: {neg.mean() * 100:.1f}% (min {train['Benefit per order'].min():,.2f})")
print("Mean shipping days | profit < 0:", round(y[neg].mean(), 3), "| profit >= 0:", round(y[~neg].mean(), 3))

**Viva point:** negative profit is a genuine loss, not an error (7.4 shows the money columns agree with each other), so these rows are **kept**. Deleting them would remove real business cases.

### 7.4 Consistency checks on the money columns

In [ ]:
checks = {
    "Sales = Product Price x Quantity": np.isclose(train["Sales"], train["Product Price"] * train["Order Item Quantity"], atol=0.05),
    "Sales per customer = Sales - Discount": np.isclose(train["Sales per customer"], train["Sales"] - train["Order Item Discount"], atol=0.05),
    "Discount Rate = Discount / Sales": np.isclose(train["Order Item Discount Rate"], train["Order Item Discount"] / train["Sales"], atol=0.01),
    "Discount Rate between 0 and 1": train["Order Item Discount Rate"].between(0, 1),
}
pd.Series({k: f"{v.mean() * 100:.2f}% of rows" for k, v in checks.items()}, name="holds for")

**Viva point:** the assignment asks to "handle invalid records" - this is the evidence. **Report the percentages you actually get**; if a rule holds for less than 100%, say so and explain (do not force it). It also shows Sales and Sales per customer are *computed* from other columns, which explains their high correlation in 6.3.

**Observation (write your own):**

## Step 8 - Time patterns

### 8.1 Shipping days by weekday

In [ ]:
DAYS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
by_weekday = y.groupby(train["order_dt"].dt.weekday).mean()
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar([DAYS[i] for i in by_weekday.index], by_weekday.values)
ax.set_ylabel("Mean real shipping days"); ax.set_title("Shipping days by order weekday")
ax.set_ylim(0, by_weekday.max() * 1.2)
show(fig, "06_days_by_weekday")

### 8.2 Shipping days by hour

In [ ]:
by_hour = y.groupby(train["order_dt"].dt.hour).mean()
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(by_hour.index, by_hour.values, marker="o")
ax.set_xlabel("Hour of order"); ax.set_ylabel("Mean real shipping days"); ax.set_title("Shipping days by order hour")
ax.set_ylim(0, by_hour.max() * 1.2)
show(fig, "07_days_by_hour")

### 8.3 Shipping days over time (drift check)

In [ ]:
monthly = train.groupby(train["order_dt"].dt.to_period("M")).agg(mean_days=(TARGET, "mean"), orders=(GROUP, "nunique"))
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(monthly.index.astype(str), monthly["mean_days"], marker=".")
ax.set_ylabel("Mean real shipping days"); ax.set_title("Shipping days over time (drift check)")
ax.set_xticks(range(0, len(monthly), max(1, len(monthly) // 12)))
ax.tick_params(axis="x", rotation=45)
show(fig, "08_days_over_time")
print(monthly.describe().round(2))

**Viva point:** a flat weekday/hour chart is evidence *against* those features; a clear pattern is evidence *for* them. A stable monthly line means no drift, so a grouped random split is acceptable; clear drift would call for a time-based split (mention it as a limitation).

**Observation (write your own):**

## Step 9 - Feature engineering
The transformers live in `src/features.py` (`DateFeatures`, `OrderFeatures`, `GeoFeatures`). They are in a `.py` file, not this notebook, so the trained pipeline can be pickled and reloaded by the backend.

| Transformer | New features | Why |
|---|---|---|
| `DateFeatures` | order_weekday, order_month, order_hour, order_quarter, is_weekend | Dispatch speed may depend on when the order arrives |
| `OrderFeatures` | order_items, order_total_sales, order_n_products | Bigger or more varied orders may take longer to pick and pack |
| `GeoFeatures` | is_domestic | Shipping within the customer's own country may be faster than international |

None uses the target, and each is computed from the order itself (at placement time), so none leaks.

### 9.1 Create the engineered features

In [ ]:
from src.features import DateFeatures, OrderFeatures, GeoFeatures

date_feats = DateFeatures().fit_transform(train)
order_feats = OrderFeatures().fit_transform(train)
parts = [date_feats, order_feats]

GEO_COLS = ["Customer Country", "Order Country"]
if all(c in train.columns for c in GEO_COLS):
    geo_feats = GeoFeatures().fit_transform(train)
    parts.append(geo_feats)
    print(f"is_domestic = 1 for {geo_feats['is_domestic'].mean() * 100:.1f}% of rows")
    print("Customer countries:", train["Customer Country"].unique())
    print("US / PR destinations:", [c for c in train["Order Country"].unique() if "Unidos" in c or "Puerto" in c])
else:
    print("Geo columns not present - GeoFeatures skipped")

engineered = pd.concat(parts, axis=1)
engineered.head()

**Check:** if `is_domestic` is 0% for every row, the country spellings do not match - fix `CUSTOMER_TO_ORDER_COUNTRY` in `src/features.py` using the printed spellings.

### 9.2 Strength of the engineered features

In [ ]:
DISCRETE = [c for c in engineered.columns if engineered[c].nunique() <= 24]
CONTINUOUS = [c for c in engineered.columns if c not in DISCRETE]

eng_strength = pd.concat([
    pd.DataFrame({"measure": "eta_squared", "value": [eta_squared(engineered[c], y) for c in DISCRETE]}, index=DISCRETE),
    pd.DataFrame({"measure": "spearman", "value": [engineered[c].corr(y, method="spearman") for c in CONTINUOUS]}, index=CONTINUOUS),
])
eng_strength.round(4)

**Viva point:** justify each feature with evidence. Weak features are kept for now - trees may combine them usefully - and M3's cross-validated feature selection decides what to drop. Compare these values with Shipping Mode's eta squared from 5.1.

### 9.3 Shipping days by order size

In [ ]:
by_items = y.groupby(engineered["order_items"]).agg(["mean", "count"])
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(by_items.index.astype(str), by_items["mean"])
ax.set_xlabel("Items in the order"); ax.set_ylabel("Mean real shipping days"); ax.set_title("Shipping days by order size")
ax.set_ylim(0, by_items["mean"].max() * 1.2)
show(fig, "09_days_by_order_size")

**Observation (write your own):**

## Step 10 - Plug the features into the pipeline
The **only** place the test set is touched: it is loaded and **transformed**, never explored.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

test = pd.read_csv(PROCESSED_DIR / "test_reg.csv")
X_tr, X_te = train.drop(columns=[TARGET, "order_dt"]), test.drop(columns=[TARGET])
LOW_CARD = [c for c in CATEGORICAL if train[c].nunique() <= 30]

transformers = [
    ("num", SimpleImputer(strategy="median"), NUMERIC),                       # M4 adds scaling here
    ("low_card", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                           ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), LOW_CARD),  # M3
    ("date", Pipeline([("feats", DateFeatures()), ("impute", SimpleImputer(strategy="most_frequent"))]), [DATE_COL]),
    ("order", OrderFeatures(), [GROUP, "Sales", "Product Name"]),
]
if all(c in X_tr.columns for c in GEO_COLS):
    transformers.append(("geo", GeoFeatures(), GEO_COLS))

prep = ColumnTransformer(transformers, remainder="drop")
Xt_tr = prep.fit_transform(X_tr)      # fitted on TRAIN only
Xt_te = prep.transform(X_te)          # test is only transformed
print("Train matrix:", Xt_tr.shape, "| Test matrix:", Xt_te.shape)
print("Any missing values after preprocessing:", bool(np.isnan(Xt_tr).any() or np.isnan(Xt_te).any()))
print("Engineered columns:", [n for n in prep.get_feature_names_out() if n.split("__")[0] in ("date", "order", "geo")])

**Viva point:** the pipeline learns only from training data; the test set goes through the same steps without influencing them, and the backend will do the same for a new order. This is where M2's work replaces the `date` placeholder in M1's `build_pipeline()`.

In [ ]:
# Pickle check: the backend must be able to load a pipeline that contains M2's classes
import pickle
pickle.loads(pickle.dumps(prep))
print("prep pickles and reloads OK")

## Step 11 - Key findings summary

In [ ]:
top_cat = cat_strength.head(3)
top_num = num_vs_target.head(3)
findings = f'''
TARGET          mean {y.mean():.2f} days, std {y.std():.2f}, range {y.min()}-{y.max()}
SHIPPING MODE   eta^2 = {eta_squared(train["Shipping Mode"], y):.3f}  (share of variance explained by mode alone)
TOP CATEGORICAL {", ".join(f"{c} ({v:.3f})" for c, v in top_cat["eta_squared"].items())}
TOP NUMERIC     {", ".join(f"{c} ({v:+.3f})" for c, v in top_num["spearman"].items())}
REDUNDANT PAIRS {len(high)} numeric pairs with |r| > 0.9
OUTLIERS        highest share: {outliers.index[0]} ({outliers.iloc[0]["outliers_%"]:.1f}% of rows)
NEGATIVE PROFIT {neg.mean() * 100:.1f}% of rows (kept - genuine losses)
MISSING         {int(missing.sum())} missing cells | DUPLICATE ROWS {train.duplicated().sum()}
ENGINEERED      {", ".join(engineered.columns)}
'''
print(findings)
(ROOT / "reports" / "m2_findings.txt").write_text(findings, encoding="utf-8")

**Be ready to explain:** (1) Shipping Mode is by far the strongest feature; (2) how much variation it leaves unexplained; (3) which features look weak; (4) why outliers and negative profits were kept; (5) why each engineered feature exists.

## Handover
| To | What M2 hands over |
|---|---|
| **M3** | eta squared ranking (5.1) and the high-cardinality columns; redundant pairs (6.3); engineered-feature strengths (9.2) |
| **M4** | outlier/skew table (7.1); boxplot (7.2); negative-profit decision (7.3) |
| **Everyone** | `src/features.py`; charts in `reports/figures/m2/` |

**My contribution in one sentence:** "I explored the training data to find which features relate to shipping time, checked data quality, outliers and time patterns, and built the date, order-level and geographic features as reusable pipeline transformers."